In [1]:
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1. PyTorch Dataset Definition

In [2]:
class LungNoduleDataset(Dataset):
    def __init__(self, manifest_path):
        # Load the manifest you generated in Phase 1
        self.manifest = pd.read_csv(manifest_path)

        # Filter out indeterminate nodules (Class -1) for binary classification
        self.manifest = self.manifest[self.manifest['malignancy_class'] != -1].reset_index(drop=True)

        self.tabular_features = ['subtlety', 'sphericity', 'margin', 'spiculation', 'texture']

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest.iloc[idx]

        # Load 3D image patch (Tell PyTorch these files are safe to fully load)
        patch_data = torch.load(row['patch_path'], weights_only=False)
        image_tensor = patch_data['tensor'] # Shape: (1, 64, 64, 64)

        # Load tabular features
        tab_tensor = torch.tensor(row[self.tabular_features].values.astype('float32'))

        # Target: Malignancy class (0: Benign, 1: Malignant)
        label = torch.tensor(row['malignancy_class'], dtype=torch.long)

        return image_tensor, tab_tensor, label

# 2. Hybrid Model Architecture

In [3]:
class CNN3DBranch(nn.Module):
    def __init__(self, output_dim=128):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            nn.Conv3d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2), # Output: 32x32x32

            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2), # Output: 16x16x16

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2), # Output: 8x8x8

            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2)  # Output: 4x4x4
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )

    def forward(self, x):
        x = self.conv_blocks(x)
        return self.fc(x)

class TabularMLPBranch(nn.Module):
    def __init__(self, input_dim=5, output_dim=32):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, output_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.mlp(x)

class HybridGatedFusionModel(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.cnn = CNN3DBranch(output_dim=128)
        self.mlp = TabularMLPBranch(input_dim=5, output_dim=32)

        # Gated Fusion Layer
        self.gate = nn.Sequential(
            nn.Linear(128 + 32, 128 + 32),
            nn.Sigmoid()
        )

        # Classifier Head
        self.classifier = nn.Sequential(
            nn.Linear(128 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, image, tabular):
        # 1. Extract Features
        img_features = self.cnn(image)
        tab_features = self.mlp(tabular)

        # 2. Concatenate
        combined = torch.cat((img_features, tab_features), dim=1)

        # 3. Gated Fusion
        gate_weights = self.gate(combined)
        gated_features = combined * gate_weights

        # 4. Predict
        out = self.classifier(gated_features)
        return out

# 3. Sanity Check & Single Batch Test

In [4]:
manifest_path = "/content/drive/MyDrive/Lung_Nodule_Project/processed_patches/manifest.csv"
dataset = LungNoduleDataset(manifest_path)

print(f"Total valid nodules for binary classification: {len(dataset)}")

# Create a temporary dataloader to test shapes
temp_loader = DataLoader(dataset, batch_size=4, shuffle=True)
images, tabs, labels = next(iter(temp_loader))

print(f"Image batch shape: {images.shape}  # Expected: (4, 1, 64, 64, 64)")
print(f"Tabular batch shape: {tabs.shape}  # Expected: (4, 5)")
print(f"Labels batch shape: {labels.shape} # Expected: (4)")

# Test the forward pass
model = HybridGatedFusionModel(num_classes=2)
out = model(images, tabs)
print(f"Model output shape: {out.shape}   # Expected: (4, 2)")

Total valid nodules for binary classification: 1242
Image batch shape: torch.Size([4, 1, 64, 64, 64])  # Expected: (4, 1, 64, 64, 64)
Tabular batch shape: torch.Size([4, 5])  # Expected: (4, 5)
Labels batch shape: torch.Size([4]) # Expected: (4)
Model output shape: torch.Size([4, 2])   # Expected: (4, 2)


# 4. GroupKFold Cross-Validation Setup

In [5]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
import torch.optim as optim
import numpy as np
from torch.utils.data import Subset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")

# Hyperparameters for prototyping (Keep epochs low for the 25-patient subset)
EPOCHS = 5
BATCH_SIZE = 8
LR = 0.001

df = dataset.manifest
groups = df['patient_id'].values
y = df['malignancy_class'].values

# Calculate Class Weights to handle imbalance
class_counts = np.bincount(y)
weights = 1.0 / class_counts
class_weights = torch.FloatTensor(weights).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# 5-Fold Split
gkf = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(gkf.split(df, y, groups)):
    print(f"\n--- Starting Fold {fold + 1} ---")

    # 1. Prevent Data Leakage on Tabular Features using per-fold StandardScaler
    scaler = StandardScaler()

    # Fit scaler ONLY on training tabular data, then transform both
    train_tabs = df.iloc[train_idx][dataset.tabular_features].values
    val_tabs = df.iloc[val_idx][dataset.tabular_features].values

    # Update the dataframe values for this specific fold's split
    df.loc[train_idx, dataset.tabular_features] = scaler.fit_transform(train_tabs)
    df.loc[val_idx, dataset.tabular_features] = scaler.transform(val_tabs)

    # 2. Create DataLoaders
    train_sub = Subset(dataset, train_idx)
    val_sub = Subset(dataset, val_idx)

    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=False)

    # 3. Initialize Model & Optimizer fresh for each fold
    model = HybridGatedFusionModel(num_classes=2).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    # 4. Mini Training Loop
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0

        for images, tabs, labels in train_loader:
            images, tabs, labels = images.to(device), tabs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images, tabs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)
        print(f"  Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f}")

    # Save the model weights for this fold to your Google Drive
    save_path = f"/content/drive/MyDrive/Lung_Nodule_Project/processed_patches/hybrid_model_fold{fold+1}.pt"
    torch.save(model.state_dict(), save_path)
    print(f"Fold {fold+1} complete and weights saved.")

print("\nCross-validation training on prototype subset complete!")

Training on device: cuda

--- Starting Fold 1 ---
  Epoch 1/5 | Train Loss: 0.6170
  Epoch 2/5 | Train Loss: 0.4448
  Epoch 3/5 | Train Loss: 0.4321
  Epoch 4/5 | Train Loss: 0.4262
  Epoch 5/5 | Train Loss: 0.4075
Fold 1 complete and weights saved.

--- Starting Fold 2 ---
  Epoch 1/5 | Train Loss: 0.5977
  Epoch 2/5 | Train Loss: 0.4675
  Epoch 3/5 | Train Loss: 0.4435
  Epoch 4/5 | Train Loss: 0.4334
  Epoch 5/5 | Train Loss: 0.4290
Fold 2 complete and weights saved.

--- Starting Fold 3 ---
  Epoch 1/5 | Train Loss: 0.6312
  Epoch 2/5 | Train Loss: 0.4495
  Epoch 3/5 | Train Loss: 0.4321
  Epoch 4/5 | Train Loss: 0.4108
  Epoch 5/5 | Train Loss: 0.4162
Fold 3 complete and weights saved.

--- Starting Fold 4 ---
  Epoch 1/5 | Train Loss: 0.5845
  Epoch 2/5 | Train Loss: 0.4407
  Epoch 3/5 | Train Loss: 0.4194
  Epoch 4/5 | Train Loss: 0.4029
  Epoch 5/5 | Train Loss: 0.3997
Fold 4 complete and weights saved.

--- Starting Fold 5 ---
  Epoch 1/5 | Train Loss: 0.5846
  Epoch 2/5 | Tra